# Model development: linear and polynomial regression

With a clean, understood dataset it is time to fit models. This notebook starts from the simplest
thing that could work — a straight line — and shows what adding polynomial terms buys and costs.
The point is the workflow: fit on the training set, judge on the test set, and compare against a
baseline.

## Learning objectives

By the end of this notebook you will be able to:

- fit a linear regression and read its coefficients;
- evaluate with MAE, RMSE, and R² using the shared metrics;
- compare against a baseline that predicts the mean;
- add polynomial features and explain the bias-variance trade-off;
- spot overfitting when training error keeps falling but test error does not.

## Concept

**Linear regression** finds the coefficients that minimise squared error between predictions and
the target. Each coefficient is the expected change in the target for a one-unit change in that
feature, holding the others fixed. It is fast, interpretable, and a strong baseline — but it can
only represent straight-line relationships.

**Polynomial features** expand the inputs with squares and products, letting a linear model bend.
Degree two adds `x²` and `x_i x_j`; degree three adds cubes and more interactions. More degrees
mean more flexibility and more parameters, so the model can fit noise in the training set. That is
**overfitting**: training error keeps improving while test error worsens.

Metrics: **MAE** is the average absolute error in target units; **RMSE** is the square root of mean
squared error and punishes large mistakes more; **R²** is the share of variance explained, where 0
means "no better than the mean" and 1 is perfect. Always report more than one and always on held-
out data.

## Worked example

### Prepare the data

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
import analysis
from ds_practice import load_california, set_seed, regression_metrics
from sklearn.model_selection import train_test_split

set_seed(42)
housing = analysis.add_features(load_california())
train, test = train_test_split(housing, test_size=0.2, random_state=42)
X_train, y_train = analysis.split_xy(train)
X_test, y_test = analysis.split_xy(test)
print("features:", list(X_train.columns))
print("train/test rows:", len(X_train), len(X_test))

features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'bedrooms_per_room', 'population_per_household', 'rooms_per_person']
train/test rows: 16512 4128


### A baseline

The mean predictor is the bar every model must clear. If a model cannot beat "predict the average",
it has learned nothing.

In [2]:
baseline = y_test.mean()
baseline_pred = [baseline] * len(y_test)
print("baseline (mean) metrics:", {k: round(v, 3) for k, v in regression_metrics(y_test, baseline_pred).items()})

baseline (mean) metrics: {'mae': 0.903, 'rmse': 1.145, 'r2': 0.0}


### Linear regression

`analysis.fit_linear` scales the features before fitting, which makes coefficients comparable.
We report metrics on both splits to see the gap.

In [3]:
linear = analysis.fit_linear(X_train, y_train)
print("train:", {k: round(v, 3) for k, v in analysis.evaluate(linear, X_train, y_train).items()})
print("test :", {k: round(v, 3) for k, v in analysis.evaluate(linear, X_test, y_test).items()})

coefs = pd.Series(linear.named_steps["model"].coef_, index=X_train.columns).sort_values(key=abs, ascending=False)
print("\nlargest coefficients:")
display(coefs.head(8).round(3))

train: {'mae': 0.483, 'rmse': 0.669, 'r2': 0.665}
test : {'mae': 0.487, 'rmse': 0.674, 'r2': 0.654}

largest coefficients:


Latitude                   -0.884
Longitude                  -0.827
MedInc                      0.782
rooms_per_person            0.374
population_per_household    0.287
Population                 -0.237
bedrooms_per_room           0.232
AveBedrms                  -0.203
dtype: float64

### Adding polynomial terms

We fit degrees one through three and compare training and test R². Watch for the moment the gap
grows.

In [4]:
results = []
for degree in (1, 2, 3):
    model = analysis.fit_polynomial(X_train, y_train, degree=degree)
    train_r2 = analysis.evaluate(model, X_train, y_train)["r2"]
    test_metrics = analysis.evaluate(model, X_test, y_test)
    results.append({
        "degree": degree,
        "train_r2": round(train_r2, 3),
        "test_r2": round(test_metrics["r2"], 3),
        "test_rmse": round(test_metrics["rmse"], 3),
    })
display(pd.DataFrame(results))

,degree,train_r2,test_r2,test_rmse
0,1,0.665,0.654,0.674
1,2,0.732,0.677,0.651
2,3,0.771,-13.242,4.320


### Interpreting the pattern

Degree two usually improves the test score because the true relationship bends; degree three often
adds little while widening the train/test gap. That gap is the signature of overfitting, and it is
why we judge on the test set rather than celebrating the training score.

In [5]:
linear_pred = linear.predict(X_test)
residuals = y_test - linear_pred
print("residual summary (actual - predicted):")
print(residuals.describe().round(3).to_string())
print("share of predictions within 0.5 (50k USD):", round((residuals.abs() < 0.5).mean(), 3))

residual summary (actual - predicted):
count    4128.000
mean        0.007
std         0.674
min        -3.171
25%        -0.404
50%        -0.093
75%         0.313
max         4.782
share of predictions within 0.5 (50k USD): 0.636


## Exercises

1. **Feature subset.** Fit linear regression using only `MedInc`, `AveRooms`, and the three derived
   ratios. Compare its test R² with the full model and say which dropped features mattered most.
2. **Degree sweep.** Extend the degree sweep to four and stop at the first degree whose test R²
   falls below the previous one. Report the table.
3. **Residual check.** Bin the residuals by `MedInc` quartile and report the mean residual per bin.
   What does a systematic pattern tell you about the linear model?

## Limitations

Polynomial features explode in number and rapidly become uninterpretable, and they still cannot
represent genuine discontinuities. The capped target means the model is structurally unable to
predict the most expensive block groups, which shows up as negative residuals there. A single
train/test split is noisy; the next notebook replaces it with cross-validation. Finally,
correlated features make individual coefficients unstable even when predictions are good.